# Model Evaluation Notebook

This notebook demonstrates how to run the model evaluation from the `eval_multiple.py` script within a Jupyter environment.

In [9]:
from typing import Any, Dict, List, Tuple

import hydra
import rootutils
from lightning import LightningDataModule, LightningModule, Trainer
from omegaconf import DictConfig, OmegaConf
import glob
import os
import numpy as np
import torch
import yaml

# Set environment variables and precision
os.environ['HYDRA_FULL_ERROR'] = '1'
torch.set_float32_matmul_precision('highest')

# Setup root directory for the project
root_dir = rootutils.find_root(search_from=os.path.dirname(os.getcwd()), indicator=".project-root")
rootutils.setup_root(root_dir, indicator=".project-root", pythonpath=True)

from src.utils import (
    RankedLogger,
    extras,
    instantiate_loggers,
    log_hyperparameters,
    task_wrapper,
)
from src.utils.metrics import calculate_metrics

In [10]:
def mean_logits_per_oid(oids,logits,labels):
    """Mean logits per object id.

    :param dataset: Dataset to use.
    :param all_idx: All idxs.
    :param all_logits: All logits.
    :return: Mean logits per object id.
    """

    #from oids ids by taking the part before _ 
    list_oids = []
    for oid in oids:
        list_oids.append(oid.split('_')[0])

    list_oids = np.array(list_oids)
    #create a dictionary with oids,logits,labels
    dict_logits = {}
    for i in range(len(list_oids)):
        if list_oids[i] not in dict_logits:
            dict_logits[list_oids[i]] = {'logits': [], 'labels': []}
        dict_logits[list_oids[i]]['logits'].append(logits[i])
        dict_logits[list_oids[i]]['labels'].append(labels[i])
    #mean logits and labels
    for key in dict_logits:
        dict_logits[key]['logits'] = np.argmax(np.mean(np.array(dict_logits[key]['logits']), axis=0))
        dict_logits[key]['labels'] = np.mean(np.array(dict_logits[key]['labels']), axis=0)
    #create a dictionary with oids,logits,labels
    return dict_logits

    

In [11]:
from thop import profile
import time

log = RankedLogger(__name__, rank_zero_only=True)


def evaluate(cfg: DictConfig, test_type: str) -> Tuple[Dict[str, Any], Dict[str, Any]]:
    """Evaluates given checkpoint on a datamodule testset.

    :param cfg: DictConfig configuration composed by Hydra.
    :param test_type: Type of test set to evaluate.
    :return: Tuple[dict, dict] with metrics and dict with all instantiated objects.
    """
    assert cfg.ckpt_path, "Checkpoint path must be provided."
    # Disable all loggers by setting the logger config to an empty list or None
    cfg.trainer.logger = False

    cfg.data.test_set_type = test_type
    cfg.data.normalize_tab = True
    cfg.data.return_snids = True
    trainer: Trainer = hydra.utils.instantiate(cfg.trainer)

    log.info("Starting testing!")

    # Find all .ckpt files recursively under cfg.ckpt_path, expecting structure like 0/checkpoints/xxx.ckpt, 1/checkpoints/xxx.ckpt, etc.
    ckpt_files = glob.glob(os.path.join(cfg.ckpt_path, "*", "checkpoints", "*.ckpt"))
    # Remove any checkpoint files that end with 'last.ckpt'
    ckpt_files = [f for f in ckpt_files if not f.endswith('last.ckpt')]

    #filter ckpt by a condition in dirname/.hydra/config.yaml , if   data.percentage >= 1
    ckpt_files_clean = []
    for ckpt in ckpt_files:
        with open(os.path.join(os.path.dirname(os.path.dirname(ckpt)), ".hydra", "config.yaml"), 'r') as f:
            config_yaml = yaml.safe_load(f)
        data_percentage = config_yaml['data'].get('percentage', 1)
        if data_percentage >= 1:
            ckpt_files_clean.append(ckpt)
    
    #data splits
    ckpt_files = ckpt_files_clean
    data_splits = []
    for ckpt in ckpt_files:
        with open(os.path.join(os.path.dirname(os.path.dirname(ckpt)), ".hydra", "config.yaml"), 'r') as f:
            config_yaml = yaml.safe_load(f)
        data_splits.append(config_yaml['data'].get('split', 0))
    print(len(ckpt_files), len(data_splits))

    #sort data splits and ckpt_files accordingly
    data_splits, ckpt_files = zip(*sorted(zip(data_splits, ckpt_files)))

        
        

    all_targets_lcs, all_preds_lcs = [], []
    all_targets_feat, all_preds_feat = [], []
    all_targets_mix, all_preds_mix = [], []
    
    # Flag to compute model complexity only once
    complexity_computed = False
    
    for split, ckpt in enumerate(ckpt_files):
        print(f"Evaluating split {split} with checkpoint {ckpt}")
        cfg.data.split = split % 5
        datamodule: LightningDataModule = hydra.utils.instantiate(cfg.data)
        model: LightningModule = hydra.utils.instantiate(cfg.model)

        # Compute FLOPs, parameters, and inference time only once
        if not complexity_computed:
            # Get a sample input from the datamodule
            datamodule.setup('test')
            sample_batch = next(iter(datamodule.test_dataloader()))
            
            # Move model to CUDA for FLOPs computation (required for Flash Attention)
            device = f'cuda:{cfg.trainer.devices[0]}' if torch.cuda.is_available() else 'cpu'
            model = model.to(device)
            model.eval()
            
            # Prepare input tensor (adjust based on your model's input structure)
            # You may need to modify this based on your model's expected input
            input_tensor = {k: v.to(device) if isinstance(v, torch.Tensor) else v 
                          for k, v in sample_batch.items() if k != 'targets' and k != 'oid'}
            
            # Prepare output file path
            output_file = os.path.join(cfg.paths.output_dir, 'model_complexity.txt')
            
            # Compute FLOPs and parameters
            try:
                flops, params = profile(model, inputs=(input_tensor,), verbose=False)
                flops_g = flops / 1e9
                params_m = params / 1e6
                
                print(f"\n{'='*50}")
                print(f"Model Complexity:")
                print(f"FLOPs: {flops_g:.2f}G")
                print(f"Params: {params_m:.2f}M")
            except Exception as e:
                print(f"Could not compute FLOPs: {e}")
                # Fall back to counting parameters only
                params = sum(p.numel() for p in model.parameters())
                params_m = params / 1e6
                flops_g = 0
                print(f"Params (counted): {params_m:.2f}M")
            
            # Compute inference time
            with torch.no_grad():
                # Warmup
                for _ in range(10):
                    _ = model(input_tensor)
                
                # Measure
                if torch.cuda.is_available():
                    torch.cuda.synchronize()
                
                start = time.time()
                num_iterations = 100
                for _ in range(num_iterations):
                    _ = model(input_tensor)
                
                if torch.cuda.is_available():
                    torch.cuda.synchronize()
                    
                avg_time = (time.time() - start) / num_iterations
                avg_time_ms = avg_time * 1000
                
                print(f"Average Inference Time: {avg_time_ms:.2f}ms")
                print(f"{'='*50}\n")
            
            # Write to file
            with open(output_file, 'w') as f:
                f.write(f"MODEL COMPLEXITY ANALYSIS\n")
                f.write(f"{'='*70}\n\n")
                f.write(f"Experiment: {cfg.get('experiment_name', 'Unknown')}\n")
                f.write(f"Model Path: {cfg.ckpt_path}\n")
                f.write(f"Generated: {time.strftime('%Y-%m-%d %H:%M:%S')}\n\n")
                f.write(f"{'-'*70}\n")
                f.write(f"COMPUTATIONAL COMPLEXITY\n")
                f.write(f"{'-'*70}\n")
                f.write(f"FLOPs: {flops_g:.2f} G\n")
                f.write(f"Parameters: {params_m:.2f} M\n\n")
                f.write(f"{'-'*70}\n")
                f.write(f"INFERENCE PERFORMANCE\n")
                f.write(f"{'-'*70}\n")
                f.write(f"Average Inference Time: {avg_time_ms:.2f} ms\n")
                f.write(f"{'='*70}\n")
            
            print(f"✓ Model complexity saved to: {output_file}\n")
            
            complexity_computed = True
        model: LightningModule = hydra.utils.instantiate(cfg.model)
        out = trainer.predict(model=model, dataloaders=datamodule, ckpt_path=ckpt)

        model_logits_lcs,model_logits_feat,model_logits_mix, model_targets, model_oids = [], [], [], [], []

        for batch in out:
            model_targets.append(batch['targets'])
            model_logits_lcs.append(batch['logits_lc'])
            model_logits_feat.append(batch['logits_feat'])
            model_logits_mix.append(batch['logits_mix'])
            model_oids += batch['oid']
        
        if model_logits_lcs and model_logits_lcs[0] is not None:
            model_logits_lcs = torch.cat(model_logits_lcs, axis=0).to(torch.float32).detach().cpu().numpy()
        else:
            model_logits_lcs = None
        if model_logits_feat and model_logits_feat[0] is not None:
            model_logits_feat = torch.cat(model_logits_feat, axis=0).to(torch.float32).detach().cpu().numpy()
        else:
            model_logits_feat = None
        if model_logits_mix and model_logits_mix[0] is not None:
            model_logits_mix = torch.cat(model_logits_mix, axis=0).to(torch.float32).detach().cpu().numpy()
        else:
            model_logits_mix = None
        model_targets = torch.cat(model_targets, axis=0).detach().cpu().numpy()
        if model_logits_lcs is not None:
            matched_predictions_lcs = mean_logits_per_oid(oids=model_oids, logits=model_logits_lcs, labels=model_targets)
            all_targets_lcs.append(np.array([matched_predictions_lcs[key]['labels'] for key in matched_predictions_lcs]))
            all_preds_lcs.append(np.array([matched_predictions_lcs[key]['logits'] for key in matched_predictions_lcs]))
        else:
            all_preds_lcs = None
        if model_logits_feat is not None:
            matched_predictions_feat = mean_logits_per_oid(oids=model_oids, logits=model_logits_feat, labels=model_targets)
            all_targets_feat.append(np.array([matched_predictions_feat[key]['labels'] for key in matched_predictions_feat]))
            all_preds_feat.append(np.array([matched_predictions_feat[key]['logits'] for key in matched_predictions_feat]))
        else:
            all_preds_feat = None
    
        if model_logits_mix is not None:
            matched_predictions_mix = mean_logits_per_oid(oids=model_oids, logits=model_logits_mix, labels=model_targets)
            all_targets_mix.append(np.array([matched_predictions_mix[key]['labels'] for key in matched_predictions_mix]))
            all_preds_mix.append(np.array([matched_predictions_mix[key]['logits'] for key in matched_predictions_mix]))
        else:
            all_preds_mix = None
    

    if all_preds_lcs is not None:
        cm_title_lcs = cfg.get('cm_title')
        if cm_title_lcs:
            cm_title_lcs = cm_title_lcs.replace('MD-', '').replace('FT-', '')
        

        calculate_metrics(
            targets_list=all_targets_lcs,
            preds_list=all_preds_lcs,
            path=cfg.paths.output_dir,
            classes=cfg.get('classes'),
            classes_order=cfg.get('classes_order'),
            cm_title=cm_title_lcs,
            experiment_name= cfg.get('experiment_name', 'new_atat_windows'),
            all_experiments_csv= cfg.get('all_experiments_csv', None),
            baseline_path=cfg.get('baseline_path', None),
            baseline_experiment_name= cfg.get('baseline_experiment_name', None),
            modality= 'LC'
        )
    if all_preds_feat is not None:
        cm_title_feat = cfg.get('cm_title')
        if cm_title_feat:
            cm_title_feat = cm_title_feat.replace('LC-', '')
        if 'FT' not in cm_title_feat:
            cm_title_feat = cm_title_feat.replace('-MTA', '')
        calculate_metrics(
            targets_list=all_targets_feat,
            preds_list=all_preds_feat,
            path=cfg.paths.output_dir,
            classes=cfg.get('classes'),
            classes_order=cfg.get('classes_order'),
            cm_title=cm_title_feat,
            experiment_name= cfg.get('experiment_name', 'new_atat_windows'),
            all_experiments_csv= cfg.get('all_experiments_csv', None),
            baseline_path=cfg.get('baseline_path', None),
            baseline_experiment_name= cfg.get('baseline_experiment_name', None),
            modality= 'Feat'
        )
    if all_preds_mix is not None:
        calculate_metrics(
            targets_list=all_targets_mix,
            preds_list=all_preds_mix,
            path=cfg.paths.output_dir,
            classes=cfg.get('classes'),
            classes_order=cfg.get('classes_order'),
            cm_title=cfg.get('cm_title'),
            experiment_name= cfg.get('experiment_name', 'new_atat_windows'),
            all_experiments_csv= cfg.get('all_experiments_csv', None),
            baseline_path=cfg.get('baseline_path', None),
            baseline_experiment_name= cfg.get('baseline_experiment_name', None),
            modality= 'Mix'
        )


In [12]:
def eval_several_exp(exp_dict):
    name = exp_dict['name']
    cm_title = exp_dict['cm_title']
    experiment_name = exp_dict['experiment_name']
    baseline_name = exp_dict['baseline_name']
    type = exp_dict.get('type', 'lc')
    experiment_path = "../logs/" + type + '/' + name
    experiment_cfg = os.path.join(experiment_path, "multirun.yaml")
    all_experiments_path = f'../logs/eval/PaperATATPeriodic{type}Comparison.csv'
    baseline_path = f'../logs/eval/{type}_Baselines.csv'
    with open(experiment_cfg, "r") as f:
        cfg = yaml.safe_load(f)

    cfg = OmegaConf.create(cfg)
    # Remove the date/time suffix from the experiment path
    experiment_name_path = "_".join(experiment_path.split('/')[-1].split('_')[:-2])
    os.makedirs(os.path.join('/home/fsoto/Documents/SSLPeriodicLCs/logs/eval', experiment_name_path), exist_ok=True)
    cfg.paths.output_dir = os.path.join('/home/fsoto/Documents/SSLPeriodicLCs/logs/eval', experiment_name_path)
    cfg.ckpt_path = experiment_path

    cfg.classes = ['CEP', 'DSCT', 'EA', 'EB/EW', 'LPV', 'Per-Other', 'RRLab', 'RRLc', 'RSCVn']
    cfg.classes_order = ['RRLc', 'RRLab', 'CEP', 'DSCT', 'EA', 'EB/EW', 'LPV','RSCVn', 'Per-Other']
    cfg.cm_title = cm_title
    cfg.trainer.devices = [0]  # Use only one GPU for evaluation
    cfg.experiment_name = experiment_name
    cfg.all_experiments_csv = all_experiments_path
    cfg.baseline_path = baseline_path
    cfg.baseline_experiment_name = baseline_name
    cfg = OmegaConf.merge(cfg, OmegaConf.from_dotlist([]))
    evaluate(cfg, test_type='test')

In [13]:
experiments_lcs = [
    {   'type': 'lc',
        'name': 'ATAT_Long_Percentages_2026-01-18_09-09-19',
        'cm_title': 'ATAT LC-MTA',
        'experiment_name': 'ATAT LC',
        'baseline_name': 'ATAT LC'
    },
    {   'type': 'lc',
        'name': 'DiffT_Long_Percentages_2026-01-25_01-05-02',
        'cm_title': 'DiffT Mag-Time-Rate LC-MTA',
        'experiment_name': 'DiffT Mag-Time-Rate LC',
        'baseline_name': 'ATAT LC'
    },
    {   'type': 'lc',
        'name': 'DIFFT_Periodic_PE_Without_Rate_2026-01-21_04-58-36',
        'cm_title': 'DiffT Mag-Time LC-MTA',
        'experiment_name': 'DiffT_Time_Mag',
        'baseline_name': 'ATAT LC'
    },
    {
        'type': 'lc',
        'name': 'DIFFT_Periodic_PE_Only_Rate_2026-01-27_16-36-51',
        'cm_title': 'DiffT Rate LC-MTA',
        'experiment_name': 'DiffT Rate LC',
        'baseline_name': 'ATAT LC'
    }
]



In [14]:
from sklearn import base


experiments_mm = [
    #{   'type': 'multimodal',
    #    'name': 'ATAT_Long_Percentages_Multimodal_2026-01-22_17-10-25',
    #    'cm_title': 'ATAT LC-MD-FT-MTA',
    #    'experiment_name': 'ATAT MM',
    #    'baseline_name': 'ATAT MM'
    #},
    #{   'type': 'multimodal',
    #    'name': 'ATAT_Long_Percentages_PT_Multimodal_2026-01-24_09-27-05',
    #    'cm_title': 'ATAT LC-MD-FT-MTA',
    #    'experiment_name': 'ATAT MM-PT',
    #    'baseline_name': 'ATAT MM'
    #},
    #{   'type': 'multimodal',
    #    'name': 'DiffT_Long_Percentages_Multimodal_2026-01-22_11-08-06',
    #    'cm_title': 'DiffT LC-MD-FT-MTA',
    #    'experiment_name': 'DiffT MM',
    #    'baseline_name': 'DiffT MM'
   ##},
   #   'type': 'multimodal',
   #    'name': 'DiffT_Long_Percentages_PT_Multimodal_2026-01-23_13-24-45',
   #    'cm_title': 'DiffT LC-MD-FT-MTA',
   #     'experiment_name': 'DiffT MM-PT',
   #    'baseline_name': 'DiffT MM'
   #},
]

In [15]:
for exp in experiments_lcs:
    eval_several_exp( exp_dict=exp)

Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


5 5
Evaluating split 0 with checkpoint ../logs/lc/ATAT_Long_Percentages_2026-01-18_09-09-19/4/checkpoints/epoch_0076.ckpt
torch.Size([1, 199, 128]) torch.Size([1, 199, 128])

Model Complexity:
FLOPs: 233.24G
Params: 0.89M


Restoring states from the checkpoint path at ../logs/lc/ATAT_Long_Percentages_2026-01-18_09-09-19/4/checkpoints/epoch_0076.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loaded model weights from the checkpoint at ../logs/lc/ATAT_Long_Percentages_2026-01-18_09-09-19/4/checkpoints/epoch_0076.ckpt


Average Inference Time: 50.26ms

✓ Model complexity saved to: /home/fsoto/Documents/SSLPeriodicLCs/logs/eval/ATAT_Long_Percentages/model_complexity.txt

torch.Size([1, 199, 128]) torch.Size([1, 199, 128])


Predicting: |          | 0/? [00:00<?, ?it/s]

...

Predicting: |          | 0/? [00:00<?, ?it/s]

Restoring states from the checkpoint path at ../logs/lc/DIFFT_Periodic_PE_Only_Rate_2026-01-27_16-36-51/4/checkpoints/epoch_0042.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loaded model weights from the checkpoint at ../logs/lc/DIFFT_Periodic_PE_Only_Rate_2026-01-27_16-36-51/4/checkpoints/epoch_0042.ckpt


Evaluating split 4 with checkpoint ../logs/lc/DIFFT_Periodic_PE_Only_Rate_2026-01-27_16-36-51/4/checkpoints/epoch_0042.ckpt


Predicting: |          | 0/? [00:00<?, ?it/s]

Metrics saved to: ../logs/eval/PaperATATPeriodiclcComparison_LC.csv
Uploaded ../logs/eval/PaperATATPeriodiclcComparison_LC.csv to sheet: PaperATATPeriodiclcComparison_LC


In [16]:
for exp in experiments_mm:
    eval_several_exp( exp_dict=exp)